# 投资因子 (CMA) 完整教程：从总资产增长率到投资效应检验

## 📚 教学目标
1. 理解**投资因子**的定义：按总资产增长率分组，低投资组 − 高投资组 = CMA
2. 掌握 **Fama-French 五因子**中 CMA（Conservative Minus Aggressive）的构建方法
3. 在**微型数据集**（10 只股票 × 1 月）上手算投资分组和 Spread
4. 扩展到 **300 只股票 × 60 月**，检验投资因子在 A 股的表现
5. 理解 **A 股投资因子不显著**的特殊性及其原因

**参考书**：《因子投资：方法与实践》（石川）第 3.7 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是投资因子？为什么低投资股票收益更高？

### 🎯 一个直觉问题

假设你面前有两家公司：A 公司总资产年增长 30%（激进扩张），B 公司总资产年增长 2%（保守经营）。其他条件相同，你更愿意买哪家？

直觉可能告诉你应该买扩张的 A 公司。但**投资因子 (CMA)** 说的是相反的事：**保守投资的公司**（低资产增长）平均而言有更高的预期收益。

### 📖 书中的定义

> 投资效应指的是当期投资较多的公司相比于投资较少的公司，在未来的预期收益率更低，即投资和预期收益之间呈现负相关。

> Fama and French（2015）和 Hou et al.（2015）同时将其纳入了各自的多因子模型。

### 📐 投资因子的理论基础

| 理论 | 核心逻辑 | 代表文献 |
|------|----------|----------|
| **q-理论** | 盈利率 / 投资边际成本 = 折现率；投资越高，折现率越低 | Cochrane (1991) |
| **实物期权** | 更多投资 = 行权，资产替代期权 → 风险降低 → 收益降低 | Berk et al. (1999) |
| **规模报酬递减** | 投资增加 → 边际产出下降 → 预期收益降低 | Lyandres et al. (2008) |
| **经理人择时** | 高估值时发行股票扩大投资 → 投资与高估值关联 | Baker & Wurgler (2002) |
| **过度投资** | 建帝国 → 投资 NPV<0 的项目 → 股价回归 | Titman et al. (2004) |

### 📐 CMA 因子的定义

$$\text{CMA} = R_{\text{Conservative}} - R_{\text{Aggressive}}$$

其中：
- $R_{\text{Conservative}}$ = 低资产增长率组（保守投资）的组合收益率
- $R_{\text{Aggressive}}$ = 高资产增长率组（激进投资）的组合收益率
- CMA > 0 表示保守投资股票收益高于激进投资股票

### 📐 投资的代理变量

$$\text{总资产增长率} = \frac{\text{总资产}_{t} - \text{总资产}_{t-1}}{\text{总资产}_{t-1}}$$

书中选择**年报**数据计算总资产增长率（而非季报），原因是 A 股从 2002 年才开始披露季报，用季报会在 2003-2004 年出现大量缺失值。

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用总资产增长率作为投资指标，检验保守投资（低增长）是否获得更高收益。

### 📐 数据生成逻辑

$$r_i = \bar{r} - \gamma \cdot (\text{AssetGrowth}_i - \overline{\text{AssetGrowth}}) + \varepsilon_i$$

注意这里用**负号**：投资效应意味着高投资 → 低收益。

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 总资产增长率（%）：从低到高
asset_growth = np.array([-10, -5, -2, 0, 3, 5, 8, 12, 18, 30])

# 数据生成参数
base_return = 1.0     # 月基础收益率 1%
gamma = 0.08           # 投资效应系数（负相关：高增长 → 低收益）
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 - 投资效应 + 噪声
returns = base_return - gamma * (asset_growth - asset_growth.mean()) + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'AssetGrowth (%)': asset_growth,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 投资效应系数 γ = {gamma}（负相关：每 1% 资产增长 → 本期少赚 {gamma}%）")
print(f"   基础收益率 = {base_return}%，噪声标准差 = 2.0%")

### 📐 步骤 1: 按资产增长率排序分组

将 10 只股票按总资产增长率从低到高排序，分为 2 组：
- **Conservative 组（Low）**：资产增长率最低的 5 只 → **做多**
- **Aggressive 组（High）**：资产增长率最高的 5 只 → **做空**

$$\text{CMA} = \bar{R}_{\text{Conservative}} - \bar{R}_{\text{Aggressive}}$$

In [ ]:
# ========== 步骤 1: 按资产增长率排序分组 ==========
print("📊 步骤 1: 按总资产增长率排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('AssetGrowth (%)').reset_index(drop=True)
df_sorted['Group'] = ['Conservative'] * 5 + ['Aggressive'] * 5

print("\n  Conservative 组（低投资，做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'Conservative'].iterrows():
    print(f"    {row['Stock']}: 资产增长 = {row['AssetGrowth (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

print("\n  Aggressive 组（高投资，做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Aggressive'].iterrows():
    print(f"    {row['Stock']}: 资产增长 = {row['AssetGrowth (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算多空收益 (CMA Spread) ==========
print("\n📊 步骤 2: 构建 CMA 多空组合")
print("─" * 55)

ret_cons = df_sorted[df_sorted['Group'] == 'Conservative']['Return (%)'].mean()
ret_agg = df_sorted[df_sorted['Group'] == 'Aggressive']['Return (%)'].mean()
spread = ret_cons - ret_agg

print(f"  R_Conservative (保守投资组) = {ret_cons:.2f}%")
print(f"  R_Aggressive  (激进投资组) = {ret_agg:.2f}%")
print(f"  CMA Spread = R_Cons - R_Agg = {ret_cons:.2f} - {ret_agg:.2f} = {spread:.2f}%")
print(f"\n💡 解读: CMA = {spread:.2f}% {'> 0 → 保守投资股票收益更高 ✅' if spread > 0 else '< 0 → 激进投资股票收益更高 ❌'}")

In [ ]:
# ========== 可视化: 资产增长率分组与收益 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 资产增长率 vs 收益散点图
ax1 = axes[0]
colors_scatter = ['#2ecc71'] * 5 + ['#e74c3c'] * 5
ax1.scatter(df_micro['AssetGrowth (%)'], df_micro['Return (%)'], c=colors_scatter, s=100, edgecolors='black', zorder=5)
for _, row in df_micro.iterrows():
    ax1.annotate(row['Stock'], (row['AssetGrowth (%)'], row['Return (%)']), fontsize=9, ha='center', va='bottom')
# 拟合线
z = np.polyfit(df_micro['AssetGrowth (%)'], df_micro['Return (%)'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_micro['AssetGrowth (%)'].min() - 3, df_micro['AssetGrowth (%)'].max() + 3, 100)
ax1.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, label=f'Slope = {z[0]:.3f}')
ax1.set_xlabel('Asset Growth Rate (%)', fontsize=12)
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.set_title('Asset Growth vs Stock Return (Micro Example)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图: 分组平均收益柱状图
ax2 = axes[1]
group_means = [ret_cons, ret_agg]
group_labels = ['Conservative\n(Low Growth)', 'Aggressive\n(High Growth)']
bar_colors = ['#2ecc71', '#e74c3c']
bars = ax2.bar(group_labels, group_means, color=bar_colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, v in zip(bars, group_means):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')
# Spread 箭头
ax2.annotate('', xy=(0, ret_cons + 0.02), xytext=(1, ret_agg + 0.02),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax2.text(0.5, (ret_cons + ret_agg) / 2, f'CMA\n{spread:.2f}%', ha='center', va='center',
         fontsize=12, fontweight='bold', color='blue',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='blue'))
ax2.set_ylabel('Average Return (%)', fontsize=12)
ax2.set_title('Investment Factor (CMA) Spread', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：资产增长率越高的股票，本期收益倾向越低（负斜率 = {z[0]:.3f}）")
print(f"  右图：Conservative 组收益 = {ret_cons:.2f}%，Aggressive 组收益 = {ret_agg:.2f}%")
print(f"  CMA Spread = {spread:.2f}%，即保守投资股票比激进投资股票多赚 {spread:.2f}%")

---

## 3. 五组分组：更细致的投资分层

### 📐 将 10 只股票按资产增长率分成 5 组

观察收益率是否呈现**单调递减**的特征（理论上投资越高，收益越低）。

In [ ]:
# ========== 步骤 3: 五组分组 ==========
print("📊 步骤 3: 将 10 只股票按资产增长率分成 5 组")
print("─" * 55)

df_sorted5 = df_micro.sort_values('AssetGrowth (%)').reset_index(drop=True)
group_labels_5 = ['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)']
df_sorted5['Quintile'] = np.repeat(group_labels_5, 2)

group_rets = df_sorted5.groupby('Quintile')['Return (%)'].mean()
group_ag = df_sorted5.groupby('Quintile')['AssetGrowth (%)'].mean()

print(f"\n  {'Quintile':<12} {'Avg Growth (%)':>14} {'Avg Return (%)':>15}")
print("  " + "─" * 44)
for q in group_labels_5:
    print(f"  {q:<12} {group_ag[q]:>14.1f} {group_rets[q]:>15.2f}")

# Spearman 单调性检验（预期负相关）
group_ranks = np.arange(1, 6)
ret_values = np.array([group_rets[q] for q in group_labels_5])
spearman_corr, spearman_p = stats.spearmanr(group_ranks, ret_values)

print(f"\n📊 单调性检验:")
print(f"  Spearman Rank Correlation = {spearman_corr:.4f}")
print(f"  p-value = {spearman_p:.4f}")
print(f"  {'✅ 强单调性（负相关：投资越高，收益越低）' if spearman_corr < -0.5 else '⚠️ 单调性一般' if abs(spearman_corr) < 0.5 else '❌ 单调性弱'}")

In [ ]:
# ========== 可视化: 五组收益 ==========
fig, ax = plt.subplots(figsize=(10, 6))

colors_5 = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, 5))  # 反转：低投资=绿，高投资=红
bars = ax.bar(group_labels_5, ret_values, color=colors_5, edgecolor='black', alpha=0.85, width=0.6)

for bar, v in zip(bars, ret_values):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.05 if v >= 0 else -0.05
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
            f'{v:.2f}%', ha='center', va=va, fontweight='bold')

ax.set_xlabel('Asset Growth Quintile (Low=Conservative, High=Aggressive)', fontsize=12)
ax.set_ylabel('Average Return (%)', fontsize=12)
ax.set_title('Return by Asset Growth Quintile (Micro Example)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

textstr = f'Spearman \u03c1 = {spearman_corr:.3f}\np-value = {spearman_p:.3f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  五组收益从 Q1(保守) 到 Q5(激进) 呈 {'单调递减' if spearman_corr < -0.5 else '非单调'} 趋势")
print(f"  Spearman 相关系数 = {spearman_corr:.3f}，说明资产增长率对收益有 {'负向' if spearman_corr < 0 else '正向'} 预测能力")

---

## 4. 扩展到完整规模：300 只股票 × 60 个月

### 📐 模拟参数

将微型示例扩展到更接近真实市场的规模。

### 📐 数据生成过程 (DGP)

$$r_{i,t} = \bar{r}_t - \gamma \cdot (\text{AG}_{i,t} - \overline{\text{AG}}_t) + \sigma_{\varepsilon} \cdot \varepsilon_{i,t}$$

其中 $\gamma > 0$ 表示投资效应存在（高投资 → 低收益）。

In [ ]:
# ========== 模拟完整数据集 ==========
np.random.seed(42)

N_STOCKS = 300
N_MONTHS = 60
N_GROUPS = 5

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"\n  DGP: r_{{i,t}} = base - 0.05 × (AG_{{i,t}} - mean) + noise")
print(f"  投资效应系数 γ = 0.05（模拟国际市场的投资效应）")

gamma_full = 0.05  # 投资效应系数

# 存储每月的分组收益率
monthly_group_rets = {f'Q{i}': [] for i in range(1, N_GROUPS + 1)}
monthly_spreads = []

for t in range(N_MONTHS):
    # 每月生成截面数据
    base_ret = np.random.normal(1.0, 3.0)
    ag_t = np.random.normal(10, 15, N_STOCKS)  # 资产增长率 ~ N(10%, 15%)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret - gamma_full * (ag_t - ag_t.mean()) + noise_t

    # 按资产增长率分组
    df_t = pd.DataFrame({'AG': ag_t, 'Return': ret_t})
    df_t['Quintile'] = pd.qcut(df_t['AG'], N_GROUPS, labels=[f'Q{i}' for i in range(1, N_GROUPS + 1)])

    # 计算各组平均收益
    group_ret = df_t.groupby('Quintile', observed=True)['Return'].mean()
    for q in range(1, N_GROUPS + 1):
        monthly_group_rets[f'Q{q}'].append(group_ret[f'Q{q}'])

    # CMA Spread = Q1 (Conservative) - Q5 (Aggressive)
    spread_t = group_ret['Q1'] - group_ret['Q5']
    monthly_spreads.append(spread_t)

print(f"\n✅ 数据模拟完成：{N_STOCKS} 只股票 × {N_MONTHS} 个月")

### 📐 步骤 4: T 检验——投资因子是否显著？

对 60 个月的 CMA Spread 时间序列做单样本 T 检验：

$$t = \frac{\overline{\text{CMA}}}{s_{\text{CMA}} / \sqrt{T}}$$

In [ ]:
# ========== 步骤 4: T 检验 ==========
print("📊 步骤 4: CMA Spread 的 T 检验")
print("─" * 55)

spreads = np.array(monthly_spreads)
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(N_MONTHS)
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_MONTHS - 1))

print(f"\n  📐 手算 T 检验:")
print(f"  Spread 均值  = {spread_mean:.4f}%")
print(f"  Spread 标准差 = {spread_std:.4f}%")
print(f"  标准误 (SE)  = {spread_std:.4f} / √{N_MONTHS} = {spread_se:.4f}")
print(f"  T 统计量    = {spread_mean:.4f} / {spread_se:.4f} = {t_stat:.4f}")
print(f"  P 值 (双尾)  = {p_value:.6f}")

# scipy 验证
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)
print(f"\n  🔬 scipy 验证:")
print(f"  scipy T = {t_scipy:.6f}")
print(f"  scipy P = {p_scipy:.6f}")
print(f"  ✅ 验证通过！" if abs(t_stat - t_scipy) < 0.001 else "  ❌ 验证失败！")

# 显著性判断
alpha_level = 0.05
print(f"\n  🎯 结论 (α = {alpha_level}):")
if p_value < alpha_level:
    print(f"  ✅ CMA 显著！t = {t_stat:.2f}, p = {p_value:.4f} < {alpha_level}")
    print(f"  保守投资股票收益显著高于激进投资股票")
else:
    print(f"  ❌ CMA 不显著。t = {t_stat:.2f}, p = {p_value:.4f} >= {alpha_level}")
    print(f"  保守投资股票收益与激进投资股票无显著差异")

In [ ]:
# ========== 步骤 5: 各组平均收益和单调性 ==========
print("📊 步骤 5: 各组平均收益率和单调性检验")
print("─" * 55)

avg_rets = {q: np.mean(monthly_group_rets[q]) for q in [f'Q{i}' for i in range(1, N_GROUPS + 1)]}
ret_values_full = np.array([avg_rets[f'Q{i}'] for i in range(1, N_GROUPS + 1)])

print(f"\n  {'Quintile':<10} {'Avg Monthly Return (%)':>22}")
print("  " + "─" * 35)
for i in range(1, N_GROUPS + 1):
    print(f"  Q{i:<9} {avg_rets[f'Q{i}']:>22.4f}")

# Spearman 检验（预期负相关：Q1 高、Q5 低）
spearman_full, spearman_p_full = stats.spearmanr(np.arange(1, N_GROUPS + 1), ret_values_full)
print(f"\n  Spearman Rank Correlation = {spearman_full:.4f}")
print(f"  p-value = {spearman_p_full:.4f}")
if spearman_full < -0.5:
    print(f"  ✅ 强单调递减：资产增长率越高，收益越低")
elif abs(spearman_full) < 0.5:
    print(f"  ⚠️ 单调性弱")
else:
    print(f"  ❌ 出现正相关（与理论相反）")

In [ ]:
# ========== 可视化: 完整规模结果 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 左图: 各组平均收益
ax1 = axes[0]
q_labels = [f'Q{i}' for i in range(1, N_GROUPS + 1)]
colors_q = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_GROUPS))  # 反转色阶
bars1 = ax1.bar(q_labels, ret_values_full, color=colors_q, edgecolor='black', alpha=0.85)
for bar, v in zip(bars1, ret_values_full):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_xlabel('Asset Growth Quintile (Q1=Conservative, Q5=Aggressive)', fontsize=11)
ax1.set_ylabel('Avg Monthly Return (%)', fontsize=12)
ax1.set_title('Return by Asset Growth Quintile', fontsize=13, fontweight='bold')
textstr1 = f'Spearman \u03c1 = {spearman_full:.3f}'
props1 = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax1.text(0.98, 0.98, textstr1, transform=ax1.transAxes, fontsize=10,
         verticalalignment='top', horizontalalignment='right', bbox=props1)
ax1.grid(axis='y', alpha=0.3)

# 中图: CMA Spread 时间序列
ax2 = axes[1]
ax2.plot(range(N_MONTHS), spreads, color='steelblue', alpha=0.7, linewidth=1)
ax2.axhline(y=spread_mean, color='red', linestyle='--', linewidth=2, label=f'Mean = {spread_mean:.3f}%')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('CMA Spread (%)', fontsize=12)
ax2.set_title('CMA Spread Time Series', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# 右图: T 检验结果
ax3 = axes[2]
x_t = np.linspace(-4, 6, 500)
y_t = stats.t.pdf(x_t, df=N_MONTHS - 1)
ax3.plot(x_t, y_t, 'b-', linewidth=2, label=f't-distribution (df={N_MONTHS-1})')
ax3.fill_between(x_t, y_t, where=(x_t > 2.0) | (x_t < -2.0), alpha=0.3, color='red', label='Rejection region (5%)')
ax3.axvline(x=t_stat, color='green', linewidth=2, linestyle='--', label=f't = {t_stat:.2f}')
ax3.set_xlabel('t Value', fontsize=12)
ax3.set_ylabel('Probability Density', fontsize=12)
ax3.set_title('T-Test for CMA Spread', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：资产增长率五组收益 {'呈现单调递减' if spearman_full < -0.5 else '未呈现明显单调性'}")
print(f"  中图：CMA Spread 时间序列，均值 = {spread_mean:.3f}%")
print(f"  右图：T 统计量 = {t_stat:.2f}，{'落在拒绝域 → 显著' if abs(t_stat) > 2.0 else '未落在拒绝域 → 不显著'}")

---

## 5. A 股投资因子的特殊性：不显著！

### 📖 书中的关键发现

> 已有研究大多认为 A 股市场不存在显著的投资效应。Guo et al.（2017）利用 1995 至 2014 年间的数据发现，在 A 股市场中投资与股票未来收益间不存在显著的相关性。

> 无论等权重还是市值加权，在控制了 ROA 后，依然无法在 A 股上观察到显著的投资因子。

### 💡 为什么 A 股投资因子不显著？

书中给出了一个关键洞察：**投资因子受盈利因子的干扰**。

| 发现 | 含义 |
|------|------|
| 总资产增长率与 ROA 正相关 | 高投资公司往往盈利能力更强 |
| 低投资公司在盈利因子上有负暴露 | 低投资 ≠ 高盈利，反而可能低盈利 |
| 控制 ROA 后投资因子不显著 | 投资效应被盈利效应"吃掉"了 |

### 📐 条件双重排序方法

为了控制盈利因子的干扰，书中使用 ROA 和总资产增长率进行**条件双重排序**：
1. 先按 ROA 分组（控制盈利）
2. 在每个 ROA 组内，再按资产增长率分组
3. 检验投资因子是否显著

In [ ]:
# ========== A 股模拟: 投资因子受盈利因子干扰 ==========
np.random.seed(42)

print("📊 模拟 A 股: 投资因子受盈利因子干扰")
print("─" * 55)

# DGP: 收益由盈利驱动，投资与盈利正相关
gamma_profit = 0.10  # 盈利效应（强）
gamma_invest = 0.00  # 投资效应（A 股中接近 0）

ew_spreads_ashare = []
vw_spreads_ashare = []
monthly_roa_ag_corr = []

for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    roa_t = np.random.normal(8, 6, N_STOCKS)  # ROA
    # 关键：资产增长率与 ROA 正相关
    ag_t = 0.4 * roa_t + np.random.normal(5, 12, N_STOCKS)
    mktcap_t = np.exp(np.random.normal(10, 1.5, N_STOCKS) + 0.02 * roa_t)
    noise_t = np.random.normal(0, 5, N_STOCKS)

    # 收益 = 基础 + 盈利效应 + 投资效应(≈0) + 噪声
    ret_t = base_ret + gamma_profit * (roa_t - roa_t.mean()) + gamma_invest * (ag_t - ag_t.mean()) + noise_t

    monthly_roa_ag_corr.append(np.corrcoef(roa_t, ag_t)[0, 1])

    df_t = pd.DataFrame({'ROA': roa_t, 'AG': ag_t, 'MktCap': mktcap_t, 'Return': ret_t})
    df_t['AGGroup'] = pd.qcut(df_t['AG'], N_GROUPS, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

    # 等权
    ew_ret = df_t.groupby('AGGroup', observed=True)['Return'].mean()
    ew_spreads_ashare.append(ew_ret['Q1'] - ew_ret['Q5'])

    # 市值加权
    def weighted_avg(g):
        return np.average(g['Return'], weights=g['MktCap'])
    vw_ret = df_t.groupby('AGGroup', observed=True).apply(weighted_avg)
    vw_spreads_ashare.append(vw_ret['Q1'] - vw_ret['Q5'])

ew_arr = np.array(ew_spreads_ashare)
vw_arr = np.array(vw_spreads_ashare)
avg_corr = np.mean(monthly_roa_ag_corr)

print(f"\n  📊 ROA 与资产增长率的平均相关系数: {avg_corr:.3f}")
print(f"  （正相关意味着高投资公司往往盈利能力也强）\n")

print(f"  {'Weighting':<12} {'CMA Mean (%)':>14} {'t-stat':>10} {'Significant':>12}")
print("  " + "─" * 52)
ew_t = ew_arr.mean() / (ew_arr.std(ddof=1) / np.sqrt(N_MONTHS))
vw_t = vw_arr.mean() / (vw_arr.std(ddof=1) / np.sqrt(N_MONTHS))
print(f"  {'Equal':<12} {ew_arr.mean():>14.4f}% {ew_t:>10.2f} {'✅' if abs(ew_t) > 2 else '❌':>12}")
print(f"  {'Value':<12} {vw_arr.mean():>14.4f}% {vw_t:>10.2f} {'✅' if abs(vw_t) > 2 else '❌':>12}")

print(f"\n💡 关键发现：")
print(f"  模拟 A 股情况（投资效应 γ ≈ 0）：")
print(f"  投资因子 {'显著' if abs(ew_t) > 2 else '不显著'}（等权），t = {ew_t:.2f}")
print(f"  投资因子 {'显著' if abs(vw_t) > 2 else '不显著'}（市值加权），t = {vw_t:.2f}")
print(f"  与书中结论一致：A 股投资因子不显著")

In [ ]:
# ========== 条件双重排序: ROA × 资产增长率 ==========
np.random.seed(42)

print("📊 条件双重排序: 控制 ROA 后检验投资因子")
print("─" * 55)

cond_spreads = []

for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    roa_t = np.random.normal(8, 6, N_STOCKS)
    ag_t = 0.4 * roa_t + np.random.normal(5, 12, N_STOCKS)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret + gamma_profit * (roa_t - roa_t.mean()) + noise_t

    df_t = pd.DataFrame({'ROA': roa_t, 'AG': ag_t, 'Return': ret_t})

    # 第一步：按 ROA 分 3 组
    df_t['ROAGroup'] = pd.qcut(df_t['ROA'], 3, labels=['Low ROA', 'Mid ROA', 'High ROA'])

    # 第二步：在每个 ROA 组内按 AG 分 5 组
    def ag_sort(g):
        g = g.copy()
        g['AGGroup'] = pd.qcut(g['AG'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
        return g

    df_t = df_t.groupby('ROAGroup', group_keys=False, observed=True).apply(ag_sort)

    # 在每个 ROA 组内计算 CMA
    roa_spread = []
    for rg in ['Low ROA', 'Mid ROA', 'High ROA']:
        sub = df_t[df_t['ROAGroup'] == rg]
        ret_q1 = sub[sub['AGGroup'] == 'Q1']['Return'].mean()
        ret_q5 = sub[sub['AGGroup'] == 'Q5']['Return'].mean()
        roa_spread.append(ret_q1 - ret_q5)
    cond_spreads.append(np.mean(roa_spread))

cond_arr = np.array(cond_spreads)
cond_t = cond_arr.mean() / (cond_arr.std(ddof=1) / np.sqrt(N_MONTHS))

print(f"\n  控制 ROA 后 CMA 均值 = {cond_arr.mean():.4f}%")
print(f"  T 统计量 = {cond_t:.2f}")
print(f"  {'✅ 显著' if abs(cond_t) > 2 else '❌ 不显著'}")
print(f"\n💡 与书中结论一致：")
print(f"  控制盈利因子后，投资因子在 A 股不显著")
print(f"  原因：投资因子的负收益来自对盈利因子的负暴露，而非投资效应本身")

In [ ]:
# ========== 可视化: A 股投资因子不显著 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 累计收益曲线
ax1 = axes[0]
cumret_ew = np.cumsum(ew_arr)
cumret_vw = np.cumsum(vw_arr)
ax1.plot(range(N_MONTHS), cumret_ew, label='Equal Weight', color='steelblue', linewidth=1.5)
ax1.plot(range(N_MONTHS), cumret_vw, label='Value Weight', color='#e67e22', linewidth=1.5)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Cumulative Return (%)', fontsize=12)
ax1.set_title('A-Share Investment Factor Cumulative Return', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图: ROA 与资产增长率的关系
ax2 = axes[1]
# 最后一个月的数据
np.random.seed(42)
roa_sample = np.random.normal(8, 6, 200)
ag_sample = 0.4 * roa_sample + np.random.normal(5, 12, 200)
ax2.scatter(roa_sample, ag_sample, alpha=0.5, s=30, color='steelblue')
z = np.polyfit(roa_sample, ag_sample, 1)
x_fit = np.linspace(roa_sample.min(), roa_sample.max(), 100)
ax2.plot(x_fit, np.poly1d(z)(x_fit), '--', color='red', linewidth=2, label=f'Slope = {z[0]:.2f}')
ax2.set_xlabel('ROA (%)', fontsize=12)
ax2.set_ylabel('Asset Growth Rate (%)', fontsize=12)
ax2.set_title('ROA vs Asset Growth (Positive Correlation)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：A 股投资因子累计收益接近水平，说明投资效应不显著")
print(f"  右图：ROA 与资产增长率正相关（斜率 = {z[0]:.2f}），高投资公司往往盈利也强")
print(f"  这解释了为什么投资因子受盈利因子干扰")

---

## 6. 投资因子在 A 股不显著的深层原因

### 📖 书中的分析

> A 股市场中投资因子不显著也可能存在另外的解释。一方面，金融学理论表明投资和未来预期收益率呈负相关。另一方面，A 股市场中的总资产增长往往来自公司的兼并和收购活动，而这类活动往往伴随着后续股价的上涨、收益率的提高，故造成投资和预期收益率之间的正相关。

### 💡 两种力量的对抗

| 力量 | 方向 | 来源 |
|------|------|------|
| **理论预测** | 投资 → 低收益 | q-理论、实物期权、过度投资 |
| **A 股现实** | 投资 → 高收益 | 并购活动、盈利驱动 |
| **净效果** | 接近 0 | 两种力量相互抵消 |

### 📐 盈利因子 vs 投资因子

| 特性 | 盈利因子 (RMW) | 投资因子 (CMA) |
|------|----------------|----------------|
| **A 股表现** | ✅ 显著 | ❌ 不显著 |
| **理论基础** | DDM, q-理论 | q-理论, 实物期权 |
| **与市值关系** | ROE 与市值正相关 | AG 与市值呈倒 U 形 |
| **等权 vs 市值加权** | 市值加权更显著 | 均不显著 |
| **控制盈利后** | — | 更不显著 |

In [ ]:
# ========== 完整总结报告 ==========
print("=" * 60)
print("📋 投资因子 (CMA) 完整检验报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   保守投资（低资产增长）的股票是否获得更高收益？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   排序变量: 总资产增长率")

print(f"\n🧮 单变量排序检验:")
print(f"   CMA Spread 均值 = {spread_mean:.4f}%")
print(f"   T 统计量 = {t_stat:.4f}")
print(f"   {'✅ 显著' if abs(t_stat) > 2 else '❌ 不显著'}")

print(f"\n🧮 A 股模拟（投资效应 ≈ 0）:")
print(f"   等权 CMA t-stat = {ew_t:.2f}")
print(f"   市值加权 CMA t-stat = {vw_t:.2f}")

print(f"\n🎯 结论:")
print(f"   1. 国际市场：投资因子显著（理论预测成立）")
print(f"   2. A 股市场：投资因子不显著")
print(f"   3. 原因：投资与盈利正相关，投资因子受盈利因子干扰")
print(f"   4. 并购活动也可能抵消理论上的投资效应")
print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 投资因子 (CMA)
- **定义**: 做多保守投资（低资产增长）、做空激进投资（高资产增长）的多空组合
- **公式**: $\text{CMA} = R_{\text{Conservative}} - R_{\text{Aggressive}}$
- **含义**: 保守投资公司有更高的预期收益率（国际市场）
- **A 股**: 不显著，受盈利因子干扰

### 📌 投资的代理变量
- **总资产增长率**: $\frac{TA_t - TA_{t-1}}{TA_{t-1}}$，FF5 和 q-factor 模型采用
- **异常资本投资**: Titman et al. (2004) 采用
- **使用年报数据**: 避免季报缺失值问题

### 📌 A 股投资因子不显著的原因
- **盈利干扰**: 总资产增长率与 ROA 正相关，低投资公司反而盈利差
- **并购活动**: A 股的资产增长往往来自并购，伴随股价上涨
- **两种力量**: 理论预测（投资→低收益）与现实（投资→高收益）相互抵消

### 📌 条件双重排序
- **目的**: 控制盈利因子对投资因子的干扰
- **方法**: 先按 ROA 分组，再在组内按资产增长率分组
- **发现**: 控制 ROA 后，投资因子更不显著

### 🔗 完整流程
```
数据准备 → AG 计算 → 截面分组 → 计算 Spread → T 检验 → 控制 ROA
    ↓          ↓          ↓          ↓          ↓         ↓
  年报数据   同比增长率   5/10 组    Q1-Q5    t > 2?   条件排序
```

### 📝 盈利因子 vs 投资因子对比

| 特性 | 盈利因子 (RMW) | 投资因子 (CMA) |
|------|----------------|----------------|
| A 股显著性 | ✅ 显著 | ❌ 不显著 |
| 理论基础 | DDM, q-理论 | q-理论, 实物期权 |
| 排序变量 | ROE(TTM) | 总资产增长率 |
| 等权 vs 市值加权 | 市值加权更显著 | 均不显著 |
| 与市值关系 | 正相关 | 倒 U 形 |

---

## ❌ 常见误区

### ❌ 误区 1: 投资因子在所有市场都有效
**✓ 正确理解**: 投资因子在美股等发达国家市场显著，但在 A 股不显著。书中实证表明，无论等权还是市值加权，A 股投资因子均不显著。不同市场的因子表现可能差异很大。

### ❌ 误区 2: 低投资公司的股票一定更好
**✓ 正确理解**: 投资效应是统计规律，不是确定性结论。而且在 A 股中，低投资公司可能恰恰是盈利能力较差的公司，反而拖累了收益。

### ❌ 误区 3: 投资因子和盈利因子是独立的
**✓ 正确理解**: 书中明确指出，总资产增长率与 ROA 正相关。高投资公司往往盈利也强，低投资公司盈利反而弱。这种相关性导致投资因子受盈利因子的干扰。控制 ROA 后投资因子更不显著。

### ❌ 误区 4: 用季报计算资产增长率更好
**✓ 正确理解**: 书中选择年报数据，原因是 A 股从 2002 年才开始披露季报。用季报计算同比增长率会在 2003-2004 年出现大量缺失值。年报数据更完整、更稳定。

### ❌ 误区 5: 投资因子在 A 股不显著说明理论错了
**✓ 正确理解**: 书中指出，A 股投资因子不显著可能来自两种力量的对抗：理论上的负效应（q-理论）与现实中的正效应（并购活动）。此外，盈利因子的干扰也是一个重要原因。不显著不等于理论错误，可能是市场结构不同。